# 🚀 Startup Funding Analysis & Prediction App

This notebook sets up and launches the Streamlit app (dashboard + 2 ML prediction tools) on Google Colab using an `ngrok` tunnel.

**Steps:** run each cell in order, top to bottom.


## 1. Install dependencies

In [ ]:
!pip install -q streamlit pandas numpy scikit-learn plotly pyngrok

## 2. Upload the dataset
Run this cell, then click **Choose Files** and select `startup_funding_final.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select startup_funding_final.csv

## 3. Write the Streamlit app to disk
This writes `app.py` into the Colab environment. You don't need to edit this cell.

In [ ]:
%%writefile app.py
"""
Startup Funding Analysis & Prediction App
==========================================
A Streamlit app that recreates the Startup Funding dashboard and adds three
ML-powered prediction tools:
  1. Predict Funding Amount (regression)
  2. Predict Funding Tier: Micro / Small / Mid / Large (classification)
  3. Predict Investment Type / Round (classification)

Run locally:    streamlit run app.py
Run on Colab:   see the instructions the assistant gave you.
"""

import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st
import streamlit.components.v1 as components
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# ----------------------------------------------------------------------------
# PAGE CONFIG
# ----------------------------------------------------------------------------
st.set_page_config(
    page_title="Startup Funding Analysis & Prediction",
    page_icon="🚀",
    layout="wide",
)

# ----------------------------------------------------------------------------
# GLOBAL STYLE — fonts, palette, motion. Tokens (used inline in the HTML
# components below too, since iframes can't see this page's CSS variables):
#   bg-deep #0A0F2C · bg-mid #151B4D · card #FFFFFF
#   indigo #5B6EF5 (primary) · emerald #16C79A (growth) · gold #F2B705 (premium)
#   coral #FF6B6B (alert) · text-light #EAF0FF · text-dark #1B1F3B
# ----------------------------------------------------------------------------
CUSTOM_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap');

html, body, [class*="css"] { font-family: 'Inter', sans-serif; }
[data-testid="stMarkdownContainer"] h1,
[data-testid="stMarkdownContainer"] h2,
[data-testid="stMarkdownContainer"] h3,
h1, h2, h3 { font-family: 'Space Grotesk', sans-serif !important; color: #EAF0FF !important; }
p, span, label, .stMarkdown, .stCaption { font-family: 'Inter', sans-serif; }

/* Remove Streamlit's default chrome so the page reads as a standalone site */
#MainMenu { visibility: hidden; }
footer { visibility: hidden; }
[data-testid="stHeader"] { background: transparent; }
[data-testid="stToolbar"] { background: transparent; }
[data-testid="stDecoration"] { display: none; }
[data-testid="stAppViewContainer"] .block-container,
.block-container { padding-top: 1.8rem; }

/* Ambient background: soft brand-color glows + a faint rising line-chart motif,
   stacked under the animated gradient — evokes "funding data" without a stock photo. */
.stApp {
    background-image:
        radial-gradient(circle at 12% 12%, rgba(91,110,245,0.16), transparent 38%),
        radial-gradient(circle at 88% 78%, rgba(22,199,154,0.14), transparent 38%),
        url("data:image/svg+xml;base64,PHN2ZyB4bWxucz0naHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmcnIHdpZHRoPSc0NjAnIGhlaWdodD0nMjMwJz4KPHBvbHlsaW5lIHBvaW50cz0nMCwxOTAgMzgsMTY1IDc2LDE3OCAxMTUsMTQwIDE1MywxNTUgMTkxLDExMCAyMzAsMTI4IDI2OCw4MCAzMDYsOTggMzQ1LDU1IDM4Myw3MiA0MjEsMzUgNDYwLDUwJwogIGZpbGw9J25vbmUnIHN0cm9rZT0ncmdiYSgyNTUsMjU1LDI1NSwwLjA1KScgc3Ryb2tlLXdpZHRoPScyJy8+CjxnIGZpbGw9J3JnYmEoMjU1LDI1NSwyNTUsMC4wNiknPgogIDxjaXJjbGUgY3g9JzM4JyBjeT0nMTY1JyByPSczJy8+CiAgPGNpcmNsZSBjeD0nMTE1JyBjeT0nMTQwJyByPSczJy8+CiAgPGNpcmNsZSBjeD0nMTkxJyBjeT0nMTEwJyByPSczJy8+CiAgPGNpcmNsZSBjeD0nMjY4JyBjeT0nODAnIHI9JzMnLz4KICA8Y2lyY2xlIGN4PSczNDUnIGN5PSc1NScgcj0nMycvPgogIDxjaXJjbGUgY3g9JzQyMScgY3k9JzM1JyByPSczJy8+CjwvZz4KPGcgc3Ryb2tlPSdyZ2JhKDI1NSwyNTUsMjU1LDAuMDM1KScgc3Ryb2tlLXdpZHRoPScxJz4KICA8bGluZSB4MT0nMCcgeTE9JzIzMCcgeDI9JzQ2MCcgeTI9JzIzMCcvPgogIDxsaW5lIHgxPScwJyB5MT0nMCcgeDI9JzAnIHkyPScyMzAnLz4KPC9nPgo8L3N2Zz4="),
        linear-gradient(160deg, #0A0F2C 0%, #151B4D 45%, #0A0F2C 100%);
    background-repeat: no-repeat, no-repeat, repeat, no-repeat;
    background-size: 100% 100%, 100% 100%, 460px 230px, 220% 220%;
    background-position: 0 0, 0 0, center, 0% 0%;
    animation: bgShift 24s ease-in-out infinite;
}
@keyframes bgShift {
    0% { background-position: 12% 12%, 88% 78%, center, 0% 0%; }
    50% { background-position: 12% 12%, 88% 78%, center, 100% 100%; }
    100% { background-position: 12% 12%, 88% 78%, center, 0% 0%; }
}

/* Sidebar */
[data-testid="stSidebar"] {
    background: #0A0F2C;
    border-right: 1px solid rgba(255,255,255,0.08);
}
[data-testid="stSidebar"] * { color: #EAF0FF !important; font-family: 'Inter', sans-serif; }
[data-testid="stSidebar"] h1 { font-family: 'Space Grotesk', sans-serif !important; }

/* Tabs — a horizontal row of individual pill/box sections, not underlined text */
[data-testid="stTabs"] [data-baseweb="tab-list"] {
    gap: 8px;
    background: rgba(255,255,255,0.04);
    padding: 6px;
    border-radius: 14px;
    border: 1px solid rgba(255,255,255,0.08);
}
[data-testid="stTabs"] [data-baseweb="tab-border"] { background: transparent !important; }
[data-testid="stTabs"] [data-baseweb="tab-highlight"] { display: none; }
[data-testid="stTabs"] button {
    font-family: 'Space Grotesk', sans-serif;
    font-weight: 600;
    color: #B8C2E8;
    background: transparent;
    border-radius: 10px;
    padding: 0.55rem 1.1rem;
    transition: all 0.25s ease;
}
[data-testid="stTabs"] button:hover { color: #EAF0FF; background: rgba(255,255,255,0.07); }
[data-testid="stTabs"] button[aria-selected="true"] {
    color: #0A0F2C !important;
    background: linear-gradient(135deg, #F2B705, #F7D673) !important;
    box-shadow: 0 4px 14px rgba(242,183,5,0.35);
}

/* Buttons */
.stButton button {
    background: linear-gradient(135deg, #5B6EF5, #16C79A);
    color: white;
    border: none;
    border-radius: 10px;
    padding: 0.6rem 1.4rem;
    font-weight: 600;
    font-family: 'Space Grotesk', sans-serif;
    transition: transform 0.2s ease, box-shadow 0.2s ease;
    box-shadow: 0 4px 14px rgba(91,110,245,0.35);
}
.stButton button:hover {
    transform: translateY(-3px) scale(1.02);
    box-shadow: 0 8px 22px rgba(22,199,154,0.45);
}
.stButton button:active { transform: translateY(0) scale(0.99); }

/* Card containers (st.container(key="card_...")) */
div[class*="st-key-card_"] {
    background: #FFFFFF;
    border-radius: 18px;
    padding: 1.1rem 1.2rem 0.3rem 1.2rem;
    box-shadow: 0 10px 28px rgba(10,15,44,0.35);
    animation: fadeUp 0.7s ease both;
    transition: transform 0.25s ease, box-shadow 0.25s ease;
    margin-bottom: 1rem;
}
div[class*="st-key-card_"]:hover {
    transform: translateY(-4px);
    box-shadow: 0 16px 36px rgba(10,15,44,0.45);
}
div[class*="st-key-card_dash1"] { animation-delay: 0.05s; }
div[class*="st-key-card_dash2"] { animation-delay: 0.10s; }
div[class*="st-key-card_dash3"] { animation-delay: 0.15s; }
div[class*="st-key-card_dash4"] { animation-delay: 0.20s; }
div[class*="st-key-card_dash5"] { animation-delay: 0.25s; }
div[class*="st-key-card_dash6"] { animation-delay: 0.30s; }
@keyframes fadeUp {
    from { opacity: 0; transform: translateY(18px); }
    to { opacity: 1; transform: translateY(0); }
}

/* Body copy that sits directly on the dark page background (not inside a white
   card) needs a light color — but text inside white cards must stay dark.
   The card-scoped rule below has higher specificity, so it wins there. */
[data-testid="stMarkdownContainer"] p { color: #C9D2F0; }
[data-testid="stCaptionContainer"], [data-testid="stCaptionContainer"] p { color: #8B95C9; }
div[class*="st-key-card_"] [data-testid="stMarkdownContainer"] p,
div[class*="st-key-card_"] [data-testid="stCaptionContainer"] { color: #1B1F3B; }

/* Plotly + dataframe + expander chrome.
   NOTE: no overflow:hidden here — it was clipping Plotly's zoom/fullscreen/reset
   controls, which is why "reset axes" and the fullscreen toggle looked broken. */
[data-testid="stPlotlyChart"] { border-radius: 12px; }
[data-testid="stExpander"] {
    background: #F6F7FC;
    border-radius: 14px;
    border: 1px solid rgba(0,0,0,0.05);
}
[data-testid="stDataFrame"] { border-radius: 12px; overflow: hidden; }
[data-testid="stAlert"] { border-radius: 14px; animation: fadeUp 0.5s ease both; }

/* Inputs */
[data-baseweb="select"] > div, .stNumberInput input {
    border-radius: 10px !important;
    font-family: 'Inter', sans-serif;
}

@media (prefers-reduced-motion: reduce) {
    .stApp, div[class*="st-key-card_"], [data-testid="stAlert"] { animation: none !important; }
}
</style>
"""
st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

DATA_PATH = "startup_funding_final.csv"

# Funding tier cutoffs, chosen on real dollar thresholds rather than quantiles so the
# labels stay interpretable: a "Large" round genuinely means $50M+, not "top 25%".
TIER_BINS = [0, 1_000_000, 10_000_000, 50_000_000, np.inf]
TIER_LABELS = ["Micro", "Small", "Mid", "Large"]
TIER_DESCRIPTIONS = {
    "Micro": "< $1M",
    "Small": "$1M – $10M",
    "Mid": "$10M – $50M",
    "Large": "$50M+",
}


# ----------------------------------------------------------------------------
# DATA LOADING & CLEANING
# ----------------------------------------------------------------------------


def clean_city(x):
    if pd.isna(x):
        return "Unknown"
    x = str(x).strip().replace("\\xc2\\xa0", "").strip()
    low = x.lower()
    mapping = {
        "bangalore": "Bangalore",
        "bengaluru": "Bangalore",
        "gurgaon": "Gurgaon",
        "gurugram": "Gurgaon",
        "new delhi": "New Delhi",
        "delhi": "New Delhi",
    }
    for k, v in mapping.items():
        if k in low:
            return v
    if low in ("unknown", "", "nan"):
        return "Unknown"
    return x.title()


def clean_industry(x):
    if pd.isna(x):
        return "Unknown"
    low = str(x).strip().lower()
    if "ecommerce" in low or "e-commerce" in low or "e commerce" in low:
        return "eCommerce"
    if "fintech" in low or "fin-tech" in low or "fin tech" in low:
        return "FinTech"
    if "edtech" in low or "ed-tech" in low or "education" in low:
        return "EdTech"
    if "consumer internet" in low:
        return "Consumer Internet"
    if "technology" in low or low == "it" or "information technology" in low:
        return "Technology"
    if "health" in low:
        return "Healthcare"
    if "logistic" in low:
        return "Logistics"
    if "food" in low:
        return "Food & Beverage"
    if "transport" in low:
        return "Transportation"
    if "real estate" in low:
        return "Real Estate"
    if "finance" in low:
        return "Finance"
    if low in ("unknown", "", "nan", "others"):
        return "Unknown"
    return x.strip().title()


def clean_investment_type(x):
    if pd.isna(x):
        return "Other"
    low = str(x).strip().lower()
    if "seed" in low or "angel" in low:
        return "Seed/Angel Funding"
    if "private equity" in low:
        return "Private Equity"
    if "series a" in low or "pre-series a" in low or "pre series a" in low:
        return "Series A"
    if "series b" in low:
        return "Series B"
    if "series c" in low:
        return "Series C"
    if any(f"series {l}" in low for l in ["d", "e", "f", "g", "h"]):
        return "Series D+"
    if "debt" in low:
        return "Debt Funding"
    if "venture" in low:
        return "Venture Round"
    if "crowd" in low:
        return "Crowd Funding"
    return "Other"


@st.cache_data
def load_data(path):
    df = pd.read_csv(path)
    df["City_clean"] = df["City"].apply(clean_city)
    df["Industry_clean"] = df["Industry"].apply(clean_industry)
    df["InvType_clean"] = df["InvestmentType"].apply(clean_investment_type)

    top_ind = df["Industry_clean"].value_counts().head(15).index
    df["Industry_clean"] = df["Industry_clean"].where(
        df["Industry_clean"].isin(top_ind), "Other"
    )
    top_city = df["City_clean"].value_counts().head(15).index
    df["City_clean"] = df["City_clean"].where(df["City_clean"].isin(top_city), "Other")

    df["InvestorCount"] = (
        df["Investors"]
        .fillna("Unknown")
        .apply(lambda s: max(1, len([i for i in str(s).split(",") if i.strip()])))
    )

    df["FundingTier"] = pd.cut(df["FundingAmount"], bins=TIER_BINS, labels=TIER_LABELS)
    return df


# ----------------------------------------------------------------------------
# MODEL TRAINING (cached as resources so it only runs once per session)
# ----------------------------------------------------------------------------


@st.cache_resource
def train_regression_model(df):
    reg_df = df.dropna(subset=["FundingAmount"]).copy()
    reg_df["LogFunding"] = np.log1p(reg_df["FundingAmount"])

    feat_cols = ["Industry_clean", "City_clean", "InvType_clean", "Year", "Quarter", "InvestorCount"]
    cat_cols = ["Industry_clean", "City_clean", "InvType_clean"]

    X = reg_df[feat_cols]
    y = reg_df["LogFunding"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    pre = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)], remainder="passthrough"
    )
    pipe = Pipeline(
        [
            ("pre", pre),
            ("model", RandomForestRegressor(n_estimators=250, max_depth=12, random_state=42, n_jobs=-1)),
        ]
    )
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    metrics = {
        "r2": r2_score(y_test, pred),
        "mae_dollar": mean_absolute_error(np.expm1(y_test), np.expm1(pred)),
    }

    # residual spread (log space) -> used to build a rough prediction interval
    resid_std = float(np.std(y_test.values - pred))

    return pipe, metrics, resid_std, feat_cols


@st.cache_resource
def train_tier_classification_model(df):
    """Predict the FundingTier bucket using the SAME inputs as the regression
    model (industry, city, investment type, timing, investor count) — i.e. no
    funding-amount feature is used, since FundingTier is derived from it.
    Bucketing the skewed dollar target into 4 interpretable classes is a more
    learnable problem for these features than exact-dollar regression."""
    tier_df = df.dropna(subset=["FundingAmount"]).copy()

    feat_cols = ["Industry_clean", "City_clean", "InvType_clean", "Year", "Quarter", "InvestorCount"]
    cat_cols = ["Industry_clean", "City_clean", "InvType_clean"]

    X = tier_df[feat_cols]
    y = tier_df["FundingTier"].astype(str)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pre = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)], remainder="passthrough"
    )
    pipe = Pipeline(
        [
            ("pre", pre),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=250, max_depth=12, random_state=42, n_jobs=-1, class_weight="balanced"
                ),
            ),
        ]
    )
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, pred),
        "f1_macro": f1_score(y_test, pred, average="macro"),
    }

    return pipe, metrics, feat_cols


@st.cache_resource
def train_classification_model(df):
    clf_df = df.copy()
    clf_df["FundingAmount_filled"] = clf_df["FundingAmount"].fillna(clf_df["FundingAmount"].median())
    clf_df["LogFunding"] = np.log1p(clf_df["FundingAmount_filled"])

    vc = clf_df["InvType_clean"].value_counts()
    keep_classes = vc[vc >= 15].index
    clf_df = clf_df[clf_df["InvType_clean"].isin(keep_classes)]

    feat_cols = ["Industry_clean", "City_clean", "Year", "Quarter", "LogFunding", "InvestorCount"]
    cat_cols = ["Industry_clean", "City_clean"]

    X = clf_df[feat_cols]
    y = clf_df["InvType_clean"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pre = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)], remainder="passthrough"
    )
    pipe = Pipeline(
        [
            ("pre", pre),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=250, max_depth=12, random_state=42, n_jobs=-1, class_weight="balanced"
                ),
            ),
        ]
    )
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, pred),
        "f1_macro": f1_score(y_test, pred, average="macro"),
    }

    return pipe, metrics, feat_cols, sorted(y.unique())


def get_feature_importance(pipe, top_n=12):
    pre = pipe.named_steps["pre"]
    model = pipe.named_steps["model"]
    names = pre.get_feature_names_out()
    importances = model.feature_importances_
    fi = pd.DataFrame({"feature": names, "importance": importances})
    fi["feature"] = fi["feature"].str.replace("cat__", "", regex=False).str.replace(
        "remainder__", "", regex=False
    )
    return fi.sort_values("importance", ascending=False).head(top_n)


# ----------------------------------------------------------------------------
# VISUAL HELPERS — consistent Plotly theme + the hero/ticker/KPI HTML widgets.
# (components.html renders in a sandboxed iframe, so colors are hardcoded here
# rather than reused from CUSTOM_CSS, which that iframe can't see.)
# ----------------------------------------------------------------------------
PALETTE = ["#5B6EF5", "#16C79A", "#F2B705", "#FF6B6B", "#8B5CF6", "#22D3EE", "#F97316", "#10B981"]


def style_fig(fig, height=380):
    fig.update_layout(
        font=dict(family="Inter, sans-serif", color="#1B1F3B", size=12),
        title_font=dict(family="Space Grotesk, sans-serif", size=16, color="#1B1F3B"),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=10, r=10, t=55, b=10),
        height=height,
        colorway=PALETTE,
        legend=dict(bgcolor="rgba(0,0,0,0)"),
    )
    fig.update_xaxes(gridcolor="rgba(0,0,0,0.07)", zerolinecolor="rgba(0,0,0,0.1)")
    fig.update_yaxes(gridcolor="rgba(0,0,0,0.07)", zerolinecolor="rgba(0,0,0,0.1)")
    return fig


def render_hero(total_funding, total_startups, total_deals, top_city, top_industry):
    ticker_text = (
        f"💰 ${total_funding/1e9:,.2f}B total funding tracked &nbsp;•&nbsp; "
        f"🏢 {total_startups:,} startups funded &nbsp;•&nbsp; "
        f"📈 {total_deals:,} deals recorded &nbsp;•&nbsp; "
        f"🌆 {top_city} leads by city &nbsp;•&nbsp; "
        f"🏭 {top_industry} leads by industry &nbsp;•&nbsp; "
    )
    html = f"""
    <html><head><style>
        html, body {{ margin:0; padding:0; background:transparent; }}
        .hero {{
            background: linear-gradient(135deg, #0A0F2C 0%, #1B2470 50%, #0A0F2C 100%);
            border-radius: 20px;
            padding: 1.8rem 2rem 1.1rem 2rem;
            font-family: 'Space Grotesk', sans-serif;
            box-sizing: border-box;
        }}
        .hero-title {{
            font-size: 2.1rem;
            font-weight: 700;
            background: linear-gradient(90deg, #EAF0FF 10%, #16C79A 55%, #F2B705 95%);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
            animation: heroFade 0.8s ease both;
            margin: 0;
        }}
        .hero-sub {{
            color: #B8C2E8;
            font-family: 'Inter', sans-serif;
            font-size: 0.95rem;
            margin-top: 0.35rem;
            animation: heroFade 0.8s ease 0.15s both;
        }}
        .ticker-wrap {{
            margin-top: 1.1rem;
            overflow: hidden;
            white-space: nowrap;
            border-top: 1px solid rgba(255,255,255,0.15);
            border-bottom: 1px solid rgba(255,255,255,0.15);
            padding: 0.5rem 0;
        }}
        .ticker-move {{
            display: inline-block;
            font-family: 'JetBrains Mono', monospace;
            font-size: 0.82rem;
            color: #EAF0FF;
            animation: tickerScroll 26s linear infinite;
            padding-left: 100%;
        }}
        @keyframes heroFade {{ from {{ opacity:0; transform:translateY(10px); }} to {{ opacity:1; transform:translateY(0); }} }}
        @keyframes tickerScroll {{ from {{ transform: translateX(0); }} to {{ transform: translateX(-100%); }} }}
        @media (prefers-reduced-motion: reduce) {{ .ticker-move {{ animation: none; }} }}
    </style>
    <link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@700&family=Inter:wght@400&family=JetBrains+Mono:wght@400&display=swap" rel="stylesheet">
    </head><body>
        <div class="hero">
            <div class="hero-title">STARTUP FUNDING INTELLIGENCE</div>
            <div class="hero-sub">Tracking India's startup funding landscape — deals, dollars, and the investors behind them.</div>
            <div class="ticker-wrap">
                <div class="ticker-move">{ticker_text}{ticker_text}</div>
            </div>
        </div>
    </body></html>
    """
    components.html(html, height=175)


def render_kpi_cards(total_funding, total_investors, total_deals, total_startups):
    cards = [
        ("💰", "Total Funding", total_funding / 1e9, "$", "bn", 2),
        ("🧑‍💼", "Total Investors", total_investors, "", "", 0),
        ("🤝", "Total Deals", total_deals, "", "", 0),
        ("🚀", "Total Startups", total_startups, "", "", 0),
    ]
    cards_html = ""
    scripts = ""
    for i, (icon, label, value, prefix, suffix, decimals) in enumerate(cards):
        cards_html += f"""
        <div class="kpi-card" style="animation-delay:{i * 0.08}s">
            <div class="kpi-icon">{icon}</div>
            <div class="kpi-label">{label}</div>
            <div class="kpi-value" id="kpi{i}">{prefix}0{suffix}</div>
        </div>
        """
        scripts += f'animateValue("kpi{i}", {value}, "{prefix}", "{suffix}", {decimals}, 1100);\n'

    html = f"""
    <html><head><style>
        html, body {{ margin:0; padding:0; background:transparent; }}
        .kpi-row {{ display:flex; gap:1rem; font-family:'Space Grotesk', sans-serif; }}
        .kpi-card {{
            flex:1; background:#FFFFFF; border-radius:16px;
            padding:1rem 1.2rem; box-shadow:0 10px 26px rgba(10,15,44,0.28);
            animation: kpiUp 0.6s ease both;
            transition: transform .25s ease, box-shadow .25s ease;
            box-sizing: border-box;
        }}
        .kpi-card:hover {{ transform: translateY(-5px); box-shadow:0 16px 34px rgba(10,15,44,0.4); }}
        .kpi-icon {{ font-size:1.5rem; }}
        .kpi-label {{
            font-size:0.74rem; color:#6B7280; font-family:'Inter',sans-serif;
            text-transform:uppercase; letter-spacing:0.06em; margin-top:0.25rem;
        }}
        .kpi-value {{ font-size:1.7rem; font-weight:700; color:#1B1F3B; margin-top:0.15rem; }}
        @keyframes kpiUp {{ from {{ opacity:0; transform:translateY(14px); }} to {{ opacity:1; transform:translateY(0); }} }}
        @media (prefers-reduced-motion: reduce) {{ .kpi-card {{ animation:none; }} }}
    </style>
    <link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@700&family=Inter:wght@400&display=swap" rel="stylesheet">
    </head><body>
        <div class="kpi-row">{cards_html}</div>
        <script>
        function animateValue(id, end, prefix, suffix, decimals, duration) {{
            const el = document.getElementById(id);
            if (!el) return;
            let startTime = null;
            function step(ts) {{
                if (!startTime) startTime = ts;
                const progress = Math.min((ts - startTime) / duration, 1);
                const eased = 1 - Math.pow(1 - progress, 3);
                const val = end * eased;
                const shown = decimals > 0 ? val.toFixed(decimals) : Math.round(val).toLocaleString();
                el.textContent = prefix + shown + suffix;
                if (progress < 1) requestAnimationFrame(step);
            }}
            requestAnimationFrame(step);
        }}
        {scripts}
        </script>
    </body></html>
    """
    components.html(html, height=130)


# ----------------------------------------------------------------------------
# LOAD DATA + TRAIN MODELS
# ----------------------------------------------------------------------------
df = load_data(DATA_PATH)
reg_pipe, reg_metrics, resid_std, reg_feat_cols = train_regression_model(df)
tier_pipe, tier_metrics, tier_feat_cols = train_tier_classification_model(df)
clf_pipe, clf_metrics, clf_feat_cols, clf_classes = train_classification_model(df)

# ----------------------------------------------------------------------------
# SIDEBAR
# ----------------------------------------------------------------------------
st.sidebar.title("🚀 Startup Funding App")
st.sidebar.markdown(
    "Explore the dashboard or use the ML tools to predict funding outcomes "
    "for a hypothetical startup."
)
year_range = st.sidebar.slider(
    "Dashboard year filter",
    int(df["Year"].min()),
    int(df["Year"].max()),
    (int(df["Year"].min()), int(df["Year"].max())),
)
inv_type_filter = st.sidebar.multiselect(
    "Dashboard investment type filter",
    sorted(df["InvType_clean"].unique()),
    default=sorted(df["InvType_clean"].unique()),
)

# ----------------------------------------------------------------------------
# TABS
# ----------------------------------------------------------------------------
tab_dash, tab_reg, tab_tier, tab_clf, tab_perf = st.tabs(
    [
        "📊 Dashboard",
        "💰 Predict Funding Amount",
        "🎯 Predict Funding Tier",
        "🏷️ Predict Investment Type",
        "📈 Model Performance",
    ]
)

# ============================== DASHBOARD ===================================
with tab_dash:
    fdf = df[
        (df["Year"] >= year_range[0])
        & (df["Year"] <= year_range[1])
        & (df["InvType_clean"].isin(inv_type_filter))
    ]

    total_funding = fdf["FundingAmount"].sum()
    total_investors = fdf["Investors"].nunique()
    total_deals = len(fdf)
    total_startups = fdf["StartupName"].nunique()
    top_city = (
        fdf.groupby("City_clean")["FundingAmount"].sum().sort_values(ascending=False).index[0]
        if len(fdf) else "—"
    )
    top_industry = (
        fdf.groupby("Industry_clean")["FundingAmount"].sum().sort_values(ascending=False).index[0]
        if len(fdf) else "—"
    )

    render_hero(total_funding, total_startups, total_deals, top_city, top_industry)
    render_kpi_cards(total_funding, total_investors, total_deals, total_startups)

    st.write("")
    col1, col2 = st.columns(2)
    with col1:
        with st.container(key="card_dash1"):
            donut_df = fdf["InvType_clean"].value_counts().reset_index()
            donut_df.columns = ["InvestmentType", "Count"]
            fig = px.pie(
                donut_df, names="InvestmentType", values="Count", hole=0.55,
                title="Count of Startups by Investment Type",
            )
            st.plotly_chart(style_fig(fig), use_container_width=True)

    with col2:
        with st.container(key="card_dash2"):
            ind_df = (
                fdf.groupby("Industry_clean")["FundingAmount"].sum().sort_values(ascending=False).head(10)
            ).reset_index()
            fig = px.bar(
                ind_df, x="FundingAmount", y="Industry_clean", orientation="h",
                title="Sum of Funding Amount by Industry",
                labels={"FundingAmount": "Funding Amount ($)", "Industry_clean": "Industry"},
            )
            fig.update_yaxes(autorange="reversed")
            st.plotly_chart(style_fig(fig), use_container_width=True)

    col3, col4 = st.columns(2)
    with col3:
        with st.container(key="card_dash3"):
            city_df = (
                fdf.groupby("City_clean")["FundingAmount"].sum().sort_values(ascending=False).head(10)
            ).reset_index()
            fig = px.bar(
                city_df, x="FundingAmount", y="City_clean", orientation="h",
                title="Sum of Funding Amount by City",
                labels={"FundingAmount": "Funding Amount ($)", "City_clean": "City"},
            )
            fig.update_yaxes(autorange="reversed")
            st.plotly_chart(style_fig(fig), use_container_width=True)

    with col4:
        with st.container(key="card_dash4"):
            year_df = fdf.groupby("Year")["FundingAmount"].sum().reset_index()
            fig = px.line(
                year_df, x="Year", y="FundingAmount", markers=True,
                title="Sum of Funding Amount by Year",
            )
            fig.update_traces(line=dict(width=3))
            st.plotly_chart(style_fig(fig), use_container_width=True)

    with st.container(key="card_dash5"):
        treemap_df = fdf.groupby("Industry_clean")["FundingAmount"].sum().reset_index()
        treemap_df = treemap_df[treemap_df["FundingAmount"] > 0]
        fig = px.treemap(
            treemap_df, path=["Industry_clean"], values="FundingAmount",
            title="Funding Amount Treemap by Industry",
        )
        st.plotly_chart(style_fig(fig, height=420), use_container_width=True)

    with st.container(key="card_dash6"):
        tier_counts = (
            fdf.dropna(subset=["FundingTier"])["FundingTier"]
            .value_counts()
            .reindex(TIER_LABELS)
            .reset_index()
        )
        tier_counts.columns = ["FundingTier", "Count"]
        fig = px.bar(
            tier_counts, x="FundingTier", y="Count", color="FundingTier",
            title="Count of Deals by Funding Tier (Micro < $1M, Small $1-10M, Mid $10-50M, Large $50M+)",
            category_orders={"FundingTier": TIER_LABELS},
            color_discrete_map={"Micro": "#5B6EF5", "Small": "#16C79A", "Mid": "#F2B705", "Large": "#FF6B6B"},
        )
        fig.update_layout(showlegend=False)
        st.plotly_chart(style_fig(fig), use_container_width=True)

    with st.expander("View filtered raw data"):
        st.dataframe(fdf.head(200))

# ============================ PREDICT FUNDING ================================
with tab_reg:
    st.title("💰 Predict Funding Amount")
    st.markdown(
        "Estimate how much funding a startup with these characteristics is "
        "likely to raise, based on patterns in historical deals."
    )

    with st.container(key="card_reg_form"):
        c1, c2, c3 = st.columns(3)
        with c1:
            industry_in = st.selectbox("Industry", sorted(df["Industry_clean"].unique()), key="reg_ind")
            city_in = st.selectbox("City", sorted(df["City_clean"].unique()), key="reg_city")
        with c2:
            invtype_in = st.selectbox("Investment Type", sorted(df["InvType_clean"].unique()), key="reg_inv")
            year_in = st.number_input("Year", min_value=2015, max_value=2030, value=2020, key="reg_year")
        with c3:
            quarter_in = st.selectbox("Quarter", [1, 2, 3, 4], key="reg_q")
            investors_in = st.number_input("Number of investors", min_value=1, max_value=20, value=1, key="reg_invc")

        predict_reg_clicked = st.button("Predict Funding Amount", type="primary")

    if predict_reg_clicked:
        X_new = pd.DataFrame(
            [{
                "Industry_clean": industry_in,
                "City_clean": city_in,
                "InvType_clean": invtype_in,
                "Year": year_in,
                "Quarter": quarter_in,
                "InvestorCount": investors_in,
            }]
        )
        log_pred = reg_pipe.predict(X_new)[0]
        pred_amount = np.expm1(log_pred)
        low = np.expm1(log_pred - resid_std)
        high = np.expm1(log_pred + resid_std)

        st.success(f"### Estimated Funding: ${pred_amount:,.0f}")
        st.caption(f"Likely range: ${low:,.0f} – ${high:,.0f} (based on model uncertainty)")

# ============================ PREDICT FUNDING TIER ================================
with tab_tier:
    st.title("🎯 Predict Funding Tier")
    st.markdown(
        "Classify a startup into a funding **tier** rather than guessing an exact dollar "
        "figure — Micro (<$1M), Small ($1-10M), Mid ($10-50M), or Large ($50M+). This uses "
        "the same inputs as the dollar prediction above, but bucketing the highly skewed "
        "funding amounts into 4 classes is usually an easier, more reliable problem for "
        "these features than predicting an exact number."
    )

    with st.container(key="card_tier_form"):
        c1, c2, c3 = st.columns(3)
        with c1:
            industry_in3 = st.selectbox("Industry", sorted(df["Industry_clean"].unique()), key="tier_ind")
            city_in3 = st.selectbox("City", sorted(df["City_clean"].unique()), key="tier_city")
        with c2:
            invtype_in3 = st.selectbox("Investment Type", sorted(df["InvType_clean"].unique()), key="tier_inv")
            year_in3 = st.number_input("Year", min_value=2015, max_value=2030, value=2020, key="tier_year")
        with c3:
            quarter_in3 = st.selectbox("Quarter", [1, 2, 3, 4], key="tier_q")
            investors_in3 = st.number_input("Number of investors", min_value=1, max_value=20, value=1, key="tier_invc")

        predict_tier_clicked = st.button("Predict Funding Tier", type="primary")

    if predict_tier_clicked:
        X_new3 = pd.DataFrame(
            [{
                "Industry_clean": industry_in3,
                "City_clean": city_in3,
                "InvType_clean": invtype_in3,
                "Year": year_in3,
                "Quarter": quarter_in3,
                "InvestorCount": investors_in3,
            }]
        )
        pred_tier = tier_pipe.predict(X_new3)[0]
        proba3 = tier_pipe.predict_proba(X_new3)[0]
        proba_df3 = pd.DataFrame({"FundingTier": tier_pipe.classes_, "Probability": proba3})
        proba_df3["FundingTier"] = pd.Categorical(proba_df3["FundingTier"], categories=TIER_LABELS, ordered=True)
        proba_df3 = proba_df3.sort_values("FundingTier")

        st.success(f"### Predicted Tier: {pred_tier} ({TIER_DESCRIPTIONS[pred_tier]})")
        with st.container(key="card_tier_result"):
            fig = px.bar(
                proba_df3, x="FundingTier", y="Probability", color="FundingTier",
                title="Predicted Probability by Tier",
                category_orders={"FundingTier": TIER_LABELS},
                color_discrete_map={"Micro": "#5B6EF5", "Small": "#16C79A", "Mid": "#F2B705", "Large": "#FF6B6B"},
            )
            fig.update_layout(showlegend=False)
            st.plotly_chart(style_fig(fig), use_container_width=True)

# ============================ PREDICT INV TYPE ================================
with tab_clf:
    st.title("🏷️ Predict Investment Type / Round")
    st.markdown(
        "Estimate the most likely funding round type for a startup with these "
        "characteristics."
    )

    with st.container(key="card_clf_form"):
        c1, c2, c3 = st.columns(3)
        with c1:
            industry_in2 = st.selectbox("Industry", sorted(df["Industry_clean"].unique()), key="clf_ind")
            city_in2 = st.selectbox("City", sorted(df["City_clean"].unique()), key="clf_city")
        with c2:
            year_in2 = st.number_input("Year", min_value=2015, max_value=2030, value=2020, key="clf_year")
            quarter_in2 = st.selectbox("Quarter", [1, 2, 3, 4], key="clf_q")
        with c3:
            funding_in2 = st.number_input(
                "Funding amount ($)", min_value=10000, max_value=4_000_000_000, value=1_000_000, step=10000, key="clf_amt"
            )
            investors_in2 = st.number_input("Number of investors", min_value=1, max_value=20, value=1, key="clf_invc")

        predict_clf_clicked = st.button("Predict Investment Type", type="primary")

    if predict_clf_clicked:
        X_new2 = pd.DataFrame(
            [{
                "Industry_clean": industry_in2,
                "City_clean": city_in2,
                "Year": year_in2,
                "Quarter": quarter_in2,
                "LogFunding": np.log1p(funding_in2),
                "InvestorCount": investors_in2,
            }]
        )
        pred_class = clf_pipe.predict(X_new2)[0]
        proba = clf_pipe.predict_proba(X_new2)[0]
        proba_df = pd.DataFrame({"InvestmentType": clf_pipe.classes_, "Probability": proba}).sort_values(
            "Probability", ascending=False
        )

        st.success(f"### Most Likely Round: {pred_class}")
        with st.container(key="card_clf_result"):
            fig = px.bar(
                proba_df, x="Probability", y="InvestmentType", orientation="h",
                title="Predicted Probability by Round Type",
            )
            fig.update_yaxes(autorange="reversed")
            st.plotly_chart(style_fig(fig), use_container_width=True)

# ============================ MODEL PERFORMANCE ================================
with tab_perf:
    st.title("📈 Model Performance")

    st.subheader("Funding Amount Regression")
    with st.container(key="card_perf1"):
        c1, c2 = st.columns(2)
        c1.metric("R² (test set)", f"{reg_metrics['r2']:.3f}")
        c2.metric("Mean Abs. Error", f"${reg_metrics['mae_dollar']:,.0f}")
        fi_reg = get_feature_importance(reg_pipe)
        fig = px.bar(fi_reg, x="importance", y="feature", orientation="h", title="Top Features")
        fig.update_yaxes(autorange="reversed")
        st.plotly_chart(style_fig(fig), use_container_width=True)

    st.subheader("Funding Tier Classification")
    with st.container(key="card_perf2"):
        c1, c2 = st.columns(2)
        c1.metric("Accuracy (test set)", f"{tier_metrics['accuracy']:.3f}")
        c2.metric("F1 Macro", f"{tier_metrics['f1_macro']:.3f}")
        fi_tier = get_feature_importance(tier_pipe)
        fig = px.bar(fi_tier, x="importance", y="feature", orientation="h", title="Top Features")
        fig.update_yaxes(autorange="reversed")
        st.plotly_chart(style_fig(fig), use_container_width=True)
        st.caption(
            "Same input features as the regression model above, but bucketed into 4 tiers "
            "instead of predicting an exact dollar figure — typically more reliable given how "
            "skewed funding amounts are."
        )

    st.subheader("Investment Type Classification")
    with st.container(key="card_perf3"):
        c1, c2 = st.columns(2)
        c1.metric("Accuracy (test set)", f"{clf_metrics['accuracy']:.3f}")
        c2.metric("F1 Macro", f"{clf_metrics['f1_macro']:.3f}")
        fi_clf = get_feature_importance(clf_pipe)
        fig = px.bar(fi_clf, x="importance", y="feature", orientation="h", title="Top Features")
        fig.update_yaxes(autorange="reversed")
        st.plotly_chart(style_fig(fig), use_container_width=True)

    st.info(
        "Models are trained fresh each time the app restarts (cached for the session). "
        "The dataset has many missing/'Undisclosed' funding amounts and very long-tail "
        "categories, so predictions are directional estimates, not precise forecasts."
    )



## 4. Set your ngrok authtoken
1. Sign up free at https://dashboard.ngrok.com/signup
2. Copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste it below.

In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

## 5. Configure Streamlit (avoids the silent hang)

On its very first run, Streamlit blocks the terminal asking for an email address — this makes the server never actually start, so the ngrok tunnel shows **"failed to load."** This cell pre-answers that prompt.

In [ ]:
import os

os.makedirs(os.path.expanduser("~/.streamlit"), exist_ok=True)
with open(os.path.expanduser("~/.streamlit/credentials.toml"), "w") as f:
    f.write('[general]\nemail=""\n')
print("Streamlit configured.")

## 6. Launch Streamlit and open the public tunnel
This cell starts the server, waits until it's actually responding (not just "a few seconds"), then opens the tunnel. If something's wrong, it prints the server log so you can see the real error.

In [ ]:
import time, subprocess, urllib.request
from pyngrok import ngrok

# kill any previous tunnels/servers from earlier attempts
ngrok.kill()
get_ipython().system_raw("pkill -f streamlit")
time.sleep(2)

# start streamlit, logging output to a file so we can debug if it fails
get_ipython().system_raw(
    "streamlit run app.py --server.port 8501 --server.headless true > /content/logs.txt 2>&1 &"
)

# actively wait until the server is really responding, instead of guessing a sleep time
server_up = False
for attempt in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen("http://localhost:8501", timeout=1)
        server_up = True
        break
    except Exception:
        continue

if not server_up:
    print("Streamlit did not start. Here are the last 40 lines of its log:\n")
    get_ipython().system("tail -n 40 /content/logs.txt")
else:
    public_url = ngrok.connect(8501)
    print("Your app is live at:", public_url)

## 7. (Optional) Stop the app
Run this when you're done to free up the tunnel and port.

In [ ]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw("pkill -f streamlit")
print("Stopped.")